In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/creditcard.csv")
df = df.drop_duplicates()

X = df.drop(columns=["Class", "Time"])
y = df["Class"]

print(X.shape, y.shape)

(283726, 29) (283726,)


In [9]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(name, len(part), "rows,", part.sum(), "frauds,",
          round(part.mean() * 100, 3), "% fraud")

train 170235 rows, 284 frauds, 0.167 % fraud
val 56745 rows, 94 frauds, 0.166 % fraud
test 56746 rows, 95 frauds, 0.167 % fraud


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[["Amount"]])

X_train_s = X_train.copy()
X_val_s = X_val.copy()
X_test_s = X_test.copy()

X_train_s["Amount"] = scaler.transform(X_train[["Amount"]])
X_val_s["Amount"] = scaler.transform(X_val[["Amount"]])
X_test_s["Amount"] = scaler.transform(X_test[["Amount"]])

print("train Amount mean:", round(X_train_s["Amount"].mean(), 4))
print("train Amount std :", round(X_train_s["Amount"].std(), 4))
print("test  Amount mean:", round(X_test_s["Amount"].mean(), 4))

train Amount mean: 0.0
train Amount std : 1.0
test  Amount mean: 0.0027


In [11]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, average_precision_score, roc_auc_score)

def evaluate(model, X, y, name):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    print("==", name, "==")
    print(confusion_matrix(y, preds))
    print("Precision:", round(precision_score(y, preds, zero_division=0), 3))
    print("Recall   :", round(recall_score(y, preds), 3))
    print("F1       :", round(f1_score(y, preds), 3))
    print("PR-AUC   :", round(average_precision_score(y, probs), 3))
    print("ROC-AUC  :", round(roc_auc_score(y, probs), 3))

## Baseline Model



In [13]:
from sklearn.dummy import DummyClassifier

dummy=DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_s,y_train)
evaluate(dummy,X_val_s,y_val,"Baseline: always normal")


== Baseline: always normal ==
[[56651     0]
 [   94     0]]
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.002
ROC-AUC  : 0.5


## Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
evaluate(lr, X_val_s, y_val, "Logistic Regression")

== Logistic Regression ==
[[56643     8]
 [   30    64]]
Precision: 0.889
Recall   : 0.681
F1       : 0.771
PR-AUC   : 0.801
ROC-AUC  : 0.972


In [16]:
weights = pd.Series(lr.coef_[0], index=X_train_s.columns).sort_values()
print(weights.head(5))
print(weights.tail(5))

V14   -0.747683
V10   -0.611350
V27   -0.431869
V15   -0.375115
V16   -0.271082
dtype: float64
V12    0.124894
V19    0.155644
V21    0.294282
V22    0.503521
V4     0.578547
dtype: float64


## Decision Tree


In [18]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42)
dt.fit(X_train_s, y_train)
evaluate(dt, X_val_s, y_val, "Decision Tree")

== Decision Tree ==
[[56640    11]
 [   22    72]]
Precision: 0.867
Recall   : 0.766
F1       : 0.814
PR-AUC   : 0.798
ROC-AUC  : 0.953


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                            min_samples_leaf=2, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)
evaluate(rf, X_val_s, y_val, "Random Forest")

== Random Forest ==
[[56643     8]
 [   16    78]]
Precision: 0.907
Recall   : 0.83
F1       : 0.867
PR-AUC   : 0.875
ROC-AUC  : 0.967
